# Sales Data Analysis
**Organization:** BharatCares in association with AICTE  
**College:** Government P.G. College, Dholpur  
**Student:** Shivam Kumar

## Project objective
Analyze retail sales transactions to understand revenue trends, product/category performance, customer segments, and regional performance. This notebook creates a reproducible analysis using a public sample Superstore dataset.

## Dataset
The notebook attempts to download a public sample Superstore CSV from a GitHub-hosted source. If it is unavailable, download a compatible Superstore CSV and update `DATA_URL` or place `Sample - Superstore.csv` beside this notebook.

Expected fields include Order Date, Sales, Profit, Quantity, Category, Sub-Category, Segment, Region, and State. The code normalizes common column-name variations where possible.

## Analysis workflow
1. Load and validate data.
2. Clean column names, dates, and numeric fields.
3. Create monthly sales and profit trends.
4. Rank categories, sub-categories, regions, and customer segments.
5. Visualize findings and calculate summary KPIs.
6. Export cleaned data and summary tables for reporting.

## Notes
Results depend on the dataset version successfully loaded. Run all cells and use the computed outputs—not assumptions—in your final report.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/plotly/datasets/master/superstore.csv"
LOCAL_CANDIDATES = ["Sample - Superstore.csv", "superstore.csv", "Superstore.csv"]

df = None
source_used = None
for candidate in LOCAL_CANDIDATES:
    if os.path.exists(candidate):
        df = pd.read_csv(candidate, encoding_errors="replace")
        source_used = candidate
        break

if df is None:
    try:
        df = pd.read_csv(DATA_URL, encoding_errors="replace")
        source_used = DATA_URL
    except Exception as exc:
        raise RuntimeError(
            "Could not load the sample dataset. Download a Superstore CSV, place it "
            "beside this notebook as 'Sample - Superstore.csv', then rerun."
        ) from exc

print("Dataset source:", source_used)
print("Rows, columns:", df.shape)
display(df.head())


In [ ]:
# Standardize column names for easier analysis
df.columns = [str(c).strip() for c in df.columns]
print("Original columns:", df.columns.tolist())

# Map common column-name variants to a consistent set.
normalized_lookup = {c.lower().replace("_", " ").strip(): c for c in df.columns}
aliases = {
    "order date": ["order date", "orderdate", "date"],
    "sales": ["sales", "revenue", "amount"],
    "profit": ["profit", "gross profit"],
    "quantity": ["quantity", "qty", "units"],
    "category": ["category", "product category"],
    "sub category": ["sub-category", "sub category", "subcategory"],
    "region": ["region", "sales region"],
    "segment": ["segment", "customer segment"],
    "state": ["state", "province"],
    "order id": ["order id", "orderid", "order number"]
}
rename_map = {}
for canonical, options in aliases.items():
    for option in options:
        key = option.lower().replace("_", " ").strip()
        if key in normalized_lookup:
            rename_map[normalized_lookup[key]] = canonical.title() if canonical != "sub category" else "Sub-Category"
            break
df = df.rename(columns=rename_map)

required = ["Sales"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required column(s): {missing_required}. Please use a sales dataset with a Sales/Revenue column.")

if "Order Date" in df.columns:
    df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
for col in ["Sales", "Profit", "Quantity", "Discount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
# Basic data quality checks and KPI summary
df = df.drop_duplicates().copy()
df = df.dropna(subset=["Sales"])
if "Order Date" in df.columns:
    df = df[df["Order Date"].notna()]

total_sales = df["Sales"].sum()
total_orders = df["Order ID"].nunique() if "Order ID" in df.columns else len(df)
total_profit = df["Profit"].sum() if "Profit" in df.columns else np.nan
total_quantity = df["Quantity"].sum() if "Quantity" in df.columns else np.nan
profit_margin = (total_profit / total_sales * 100) if pd.notna(total_profit) and total_sales else np.nan

kpis = pd.DataFrame({
    "Metric": ["Rows analyzed", "Total sales", "Unique orders / records", "Total profit", "Total quantity", "Profit margin (%)"],
    "Value": [len(df), total_sales, total_orders, total_profit, total_quantity, profit_margin]
})
display(kpis)


In [ ]:
# Monthly sales and profit trend
if "Order Date" in df.columns:
    df["Order Month"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
    monthly = df.groupby("Order Month", as_index=False).agg(
        Sales=("Sales", "sum"),
        **({"Profit": ("Profit", "sum")} if "Profit" in df.columns else {})
    ).sort_values("Order Month")
    display(monthly.head())

    plt.figure(figsize=(12, 5))
    plt.plot(monthly["Order Month"], monthly["Sales"], marker="o")
    plt.title("Monthly Sales Trend")
    plt.xlabel("Month")
    plt.ylabel("Sales")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    if "Profit" in monthly.columns:
        plt.figure(figsize=(12, 5))
        plt.plot(monthly["Order Month"], monthly["Profit"], marker="o")
        plt.axhline(0, linewidth=1)
        plt.title("Monthly Profit Trend")
        plt.xlabel("Month")
        plt.ylabel("Profit")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No usable Order Date column found; skipping time-series analysis.")


In [ ]:
# Category and sub-category performance
for col in ["Category", "Sub-Category", "Region", "Segment", "State"]:
    if col in df.columns:
        aggregations = {"Sales": ("Sales", "sum")}
        if "Profit" in df.columns:
            aggregations["Profit"] = ("Profit", "sum")
        if "Quantity" in df.columns:
            aggregations["Quantity"] = ("Quantity", "sum")
        summary = df.groupby(col, dropna=False).agg(**aggregations).sort_values("Sales", ascending=False)
        print(f"Performance by {col}:")
        display(summary)

        if col in ["Category", "Sub-Category", "Region", "Segment"]:
            summary["Sales"].sort_values().plot(kind="barh", figsize=(9, max(4, len(summary) * 0.35)))
            plt.title(f"Sales by {col}")
            plt.xlabel("Sales")
            plt.ylabel(col)
            plt.tight_layout()
            plt.show()


In [ ]:
# Optional: top 10 products by sales, if product-level data is available
product_col = next((c for c in ["Product Name", "Product", "Item"] if c in df.columns), None)
if product_col:
    product_sales = df.groupby(product_col)["Sales"].sum().sort_values(ascending=False).head(10)
    display(product_sales.to_frame("Sales"))
    product_sales.sort_values().plot(kind="barh", figsize=(10, 6))
    plt.title("Top 10 Products by Sales")
    plt.xlabel("Sales")
    plt.tight_layout()
    plt.show()
else:
    print("No product-name column found; skipping product-level ranking.")


In [ ]:
# Sales versus profit relationship, where profit is available
if "Profit" in df.columns:
    plt.figure(figsize=(8, 6))
    plt.scatter(df["Sales"], df["Profit"], alpha=0.45)
    plt.axhline(0, linewidth=1)
    plt.title("Sales vs Profit")
    plt.xlabel("Sales")
    plt.ylabel("Profit")
    plt.tight_layout()
    plt.show()

    print("Correlation between sales and profit:", round(df[["Sales", "Profit"]].corr().iloc[0, 1], 3))
else:
    print("No Profit column found; skipping profit analysis.")


In [ ]:
# Export analysis outputs
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/cleaned_sales_data.csv", index=False)

if "Order Month" in df.columns:
    df.groupby("Order Month", as_index=False)["Sales"].sum().to_csv(
        "outputs/monthly_sales_summary.csv", index=False
    )

for col in ["Category", "Sub-Category", "Region", "Segment"]:
    if col in df.columns:
        agg = {"Sales": "sum"}
        if "Profit" in df.columns:
            agg["Profit"] = "sum"
        df.groupby(col, dropna=False).agg(agg).sort_values("Sales", ascending=False).to_csv(
            f"outputs/sales_by_{col.lower().replace('-', '_').replace(' ', '_')}.csv"
        )

print("Exported cleaned data and available summary tables to the outputs/ folder.")


## Findings and recommendations — complete after running
Use the actual tables and charts above to record:
1. Which month/period had the highest sales?
2. Which category, sub-category, region, or segment generated the most sales?
3. Which areas had low or negative profit, if profit data is present?
4. What practical recommendations follow from the results?

Do not invent results. Findings should be directly supported by notebook outputs.
